In [1]:
# First, we import our libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.svm import SVC, SVR
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sklearn.pipeline import Pipeline

In [2]:
import ssl
from sklearn.datasets import fetch_california_housing
ssl._create_default_https_context = ssl._create_unverified_context
housing = fetch_california_housing()
ssl._create_default_https_context = ssl.create_default_context  # restore after
print(type(housing))
print(housing.data.shape if hasattr(housing, 'data') else "no data attribute")

<class 'sklearn.utils._bunch.Bunch'>
(20640, 8)


In [3]:
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target

In [5]:
X.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [7]:
y

array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894], shape=(20640,))

## TRAIN_TEST_SPLIT

In [4]:
X_train,X_test, y_train,y_test = train_test_split(X, y, test_size = 0.2, random_state=42)

In [10]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## USING LINEAR REGRESSION MODEL

In [11]:
from sklearn.linear_model import LinearRegression
linear_reg = LinearRegression()
linear_reg.fit(X_train_scaled, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [17]:
from sklearn.metrics import mean_squared_error
y_pred = linear_reg.predict(X_test_scaled)
mse = mean_squared_error(y_pred, y_test)
print(f"mean_squared_error: {mse:.2f}")

mean_squared_error: 0.56


In [18]:
linear_reg.score(X_test_scaled, y_test)

0.5757877060324508

In [28]:
def predict_house_price(MedInc, HouseAge, AveRooms, AveBedrms,
 Population, AveOccup, Latitude, Longitude):
    sample = [[MedInc, HouseAge, AveRooms, AveBedrms,
 Population, AveOccup, Latitude, Longitude]]
    sample = scaler.transform(sample)

    predicted_price = linear_reg.predict(sample)[0]
    return(f"predicted price: {predicted_price *100000:,.2f}")


In [29]:
predict_house_price(MedInc=8.3, HouseAge=41.0, AveRooms=6.0, AveBedrms=1.0,
 Population=980,AveOccup=2.5, Latitude=37.5, Longitude =-122.3)


C:\Users\DELL\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


'predicted price: 443,209.67'

## USING PIPELINE

In [6]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])
pipeline.fit(X_train, y_train)

,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None


In [7]:
y_pred = pipeline.predict(X_test)
mse = mean_squared_error(y_test,y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Mse: {mse:.2f}")
print(f"R2:{r2:.2f}")

Mse: 0.56
R2:0.58


In [8]:
def predict_house_price(MedInc, HouseAge, AveRooms, AveBedrms,
 Population, AveOccup, Latitude, Longitude):

 # Prepare input as a 2D array (one row, 8 features)
 sample = pd.DataFrame({
 'MedInc': [MedInc],
 'HouseAge': [HouseAge],
 'AveRooms': [AveRooms],
 'AveBedrms': [AveBedrms],
 'Population': [Population],
 'AveOccup': [AveOccup],
 'Latitude': [Latitude],
 'Longitude': [Longitude]
 })

 predicted_price = pipeline.predict(sample)[0]
 return f"Estimated House Price: ${predicted_price * 100_000:,.2f}"

In [9]:
predict_house_price(MedInc=8.3, HouseAge=41.0, AveRooms=6.0, AveBedrms=1.0,
 Population=980, AveOccup=2.5, Latitude=37.5, Longitude =-122.3) 

'Estimated House Price: $443,209.67'